In [1]:
!pip install -q transformers torch

In [2]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

model_name = "distilbert-base-cased-distilled-squad"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

print("Document QA model loaded successfully!")

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Document QA model loaded successfully!


In [3]:
def answer_question(question, context):
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    start_index = torch.argmax(outputs.start_logits)
    end_index = torch.argmax(outputs.end_logits)

    answer_tokens = inputs["input_ids"][0][start_index:end_index + 1]

    answer = tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    )

    return answer

In [4]:
context = """
Artificial intelligence is a branch of computer science that focuses
on creating systems capable of performing tasks that normally require
human intelligence. Machine learning is a subset of artificial
intelligence that allows computers to learn patterns from data.
Deep learning is a type of machine learning that uses neural networks
with multiple layers.
"""

question = "What is machine learning?"

answer = answer_question(question, context)

print("Context:")
print(context)

print("\nQuestion:")
print(question)

print("\nAnswer:")
print(answer)

Context:

Artificial intelligence is a branch of computer science that focuses
on creating systems capable of performing tasks that normally require
human intelligence. Machine learning is a subset of artificial
intelligence that allows computers to learn patterns from data.
Deep learning is a type of machine learning that uses neural networks
with multiple layers.


Question:
What is machine learning?

Answer:
artificial intelligence


In [5]:
context = input("Enter your document/context:\n")

print("\nDocument loaded successfully!")

while True:
    question = input("\nAsk a question (or type 'exit' to stop): ")

    if question.lower() == "exit":
        print("Document QA stopped.")
        break

    answer = answer_question(question, context)

    print("\nAnswer:")
    print(answer)

Enter your document/context:
Artificial intelligence is a field of computer science that focuses on creating intelligent systems. Machine learning is a subset of artificial intelligence that enables computers to learn from data. Deep learning is a type of machine learning that uses neural networks with multiple layers. Python is one of the most popular programming languages used in artificial intelligence and machine learning.

Document loaded successfully!

Ask a question (or type 'exit' to stop): What is machine learning?

Answer:
a subset of artificial intelligence

Ask a question (or type 'exit' to stop): exit
Document QA stopped.


In [6]:
def answer_question(question, context):
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    start_probs = torch.softmax(outputs.start_logits, dim=1)
    end_probs = torch.softmax(outputs.end_logits, dim=1)

    start_index = torch.argmax(start_probs)
    end_index = torch.argmax(end_probs)

    answer_tokens = inputs["input_ids"][0][start_index:end_index + 1]

    answer = tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    )

    confidence = (
        start_probs[0][start_index].item() *
        end_probs[0][end_index].item()
    )

    return answer, confidence

In [7]:
context = """
Artificial intelligence is a field of computer science that focuses
on creating intelligent systems. Machine learning is a subset of
artificial intelligence that enables computers to learn from data.
"""

question = "What is machine learning?"

answer, confidence = answer_question(question, context)

print("Question:", question)
print("Answer:", answer)
print(f"Confidence: {confidence:.2%}")

Question: What is machine learning?
Answer: a subset of artificial intelligence
Confidence: 26.48%


In [8]:
answer = answer_question(question, context)

In [9]:
answer, confidence = answer_question(question, context)

print("\nAnswer:")
print(answer)

print(f"Confidence: {confidence:.2%}")


Answer:
a subset of artificial intelligence
Confidence: 26.48%
